# Level 2 — Task 2: Decision Trees for Classification
**Dataset:** Iris | **Tools:** pandas, scikit-learn, matplotlib, seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")


## 2. Load & Prepare Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload iris.csv

df = pd.read_csv("iris.csv")

le = LabelEncoder()
df["species_encoded"] = le.fit_transform(df["species"])

X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Classes: {list(le.classes_)}")


## 3. Train Baseline Decision Tree (No Pruning)

In [ ]:
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

y_pred_full = dt_full.predict(X_test)

print(f"Depth         : {dt_full.get_depth()}")
print(f"Leaves        : {dt_full.get_n_leaves()}")
print(f"Train Accuracy: {dt_full.score(X_train, y_train):.4f}")
print(f"Test Accuracy : {accuracy_score(y_test, y_pred_full):.4f}")
print(f"F1 (macro)    : {f1_score(y_test, y_pred_full, average='macro'):.4f}")


## 4. Visualize Full Tree

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_full, ax=ax,
    feature_names=X.columns,
    class_names=le.classes_,
    filled=True, rounded=True, fontsize=9
)
ax.set_title("Decision Tree — No Pruning", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Prune the Tree — Find Optimal max_depth

In [ ]:
depths = range(1, 11)
train_acc, test_acc, cv_acc = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_acc.append(dt.score(X_train, y_train))
    test_acc.append(dt.score(X_test, y_test))
    cv = cross_val_score(dt, X, y, cv=5, scoring="accuracy")
    cv_acc.append(cv.mean())

optimal_depth = depths[np.argmax(cv_acc)]
print(f"Optimal max_depth (by 5-fold CV): {optimal_depth}")

plt.figure(figsize=(10, 5))
plt.plot(depths, train_acc, marker="o", label="Train Accuracy")
plt.plot(depths, test_acc,  marker="s", label="Test Accuracy")
plt.plot(depths, cv_acc,    marker="^", label="CV Accuracy (5-fold)", linestyle="--")
plt.axvline(optimal_depth, color="red", linestyle=":", linewidth=1.5, label=f"Optimal depth={optimal_depth}")
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Tree Depth")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Train Pruned Tree & Visualize

In [ ]:
dt_pruned = DecisionTreeClassifier(max_depth=optimal_depth, random_state=42)
dt_pruned.fit(X_train, y_train)

y_pred_pruned = dt_pruned.predict(X_test)

print(f"Depth         : {dt_pruned.get_depth()}")
print(f"Leaves        : {dt_pruned.get_n_leaves()}")
print(f"Train Accuracy: {dt_pruned.score(X_train, y_train):.4f}")
print(f"Test Accuracy : {accuracy_score(y_test, y_pred_pruned):.4f}")
print(f"F1 (macro)    : {f1_score(y_test, y_pred_pruned, average='macro'):.4f}")

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(
    dt_pruned, ax=ax,
    feature_names=X.columns,
    class_names=le.classes_,
    filled=True, rounded=True, fontsize=10
)
ax.set_title(f"Pruned Decision Tree (max_depth={optimal_depth})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
print("Tree structure (text):")
print(export_text(dt_pruned, feature_names=list(X.columns)))


## 7. Evaluation

In [ ]:
print("Classification Report — Pruned Tree:")
print(classification_report(y_test, y_pred_pruned, target_names=le.classes_))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_full, y_pred_pruned],
    ["Full Tree (No Pruning)", f"Pruned Tree (depth={optimal_depth})"]
):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)

plt.suptitle("Confusion Matrix Comparison", fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt_pruned.feature_importances_
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(importance_df["Feature"], importance_df["Importance"], color="#4C72B0", edgecolor="white")
plt.xlabel("Gini Importance")
plt.title("Feature Importance — Pruned Decision Tree")
plt.tight_layout()
plt.show()


## 9. Summary

| Model | Depth | Leaves | Test Accuracy | F1 (macro) |
|---|---|---|---|---|
| Full Tree | See output | See output | See output | See output |
| Pruned Tree | See output | See output | See output | See output |

Pruning reduces model complexity and prevents overfitting by limiting the depth of the tree, trading marginal training accuracy for better generalization.